# Gemma 2 vs GPT-4o-mini Comparison & Evaluation (RunPod A100)

이 노트북은 **RunPod A100-SXM** 환경에서 파인튜닝된 Gemma 2 모델과 GPT-4o-mini를 비교하고, GPT-4o를 심판으로 자동 평가하기 위해 최적화되었습니다.

### 🚀 RunPod 최적화 사항:
1. **인증**: `google.colab.userdata` 대신 환경 변수(`os.getenv`)를 사용합니다.
2. **경로**: `/content/` 대신 `/workspace/` 경로를 사용합니다.
3. **성능**: A100 전용 BF16 및 Flash Attention 2를 사용합니다.
4. **호환성**: 최신 `transformers` 라이브러리로 강제 업데이트합니다.

In [ ]:
# 1. 필수 라이브러리 설치
!pip install -q -U transformers datasets peft trl bitsandbytes accelerate python-dotenv openai

In [ ]:
# 2. 임포트 및 로그인
import os
import torch
import json
import random
import pandas as pd
from huggingface_hub import login
from openai import OpenAI
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# RunPod 환경 변수에서 토큰 읽기
hf_token = os.getenv("HF_TOKEN")
openai_api_key = os.getenv("OPENAI_API_KEY")

if hf_token: login(token=hf_token)
else: print("⚠️ HF_TOKEN 환경 변수를 설정해주세요.")

if openai_api_key: client = OpenAI(api_key=openai_api_key)
else: print("⚠️ OPENAI_API_KEY 환경 변수를 설정해주세요.")

In [ ]:
# 3. 모델 로드 (A100 최적화)
base_model_id = "google/gemma-2-9b-it"
adapter_path = "/workspace/gemma2-pet-counselor-lora" # 파인튜닝 결과물 경로

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Base Model...")
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="flash_attention_2", # A100 가속
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

print("Loading Adapter...")
model = PeftModel.from_pretrained(model, adapter_path)
print("✅ All models loaded on A100!")

In [ ]:
# 4. 응답 생성 함수 정의
def generate_gemma_response(instruction, user_input):
    prompt = (
        f"<start_of_turn>user\n"
        f"{instruction}\n\n"
        f"질문: {user_input}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("model\n")[-1].strip()

def get_gpt_mini_response(instruction, user_input):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": instruction},
            {"role": "user", "content": user_input}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content.strip()

In [ ]:
# 5. 자동 평가(GPT-4o 심판) 함수
def evaluate_with_gpt4o(question, ground_truth, model_a_ans, model_b_ans):
    prompt = f"""
    당신은 반려동물 상담 전문 챗봇의 성능을 평가하는 심판입니다.
    질문과 정답지(Ground Truth)를 참고하여, 두 모델의 답변을 [정확성, 말투, 지시이행] 기준으로 평가하세요.

    [질문]: {question}
    [정답지]: {ground_truth}

    [모델 A(Gemma 2)]: {model_a_ans}
    [모델 B(GPT-4o-mini)]: {model_b_ans}

    [결과 형식] 반드시 JSON으로 답변:
    {{
        "gemma_score": (0~10),
        "gpt_score": (0~10),
        "winner": "Gemma" or "GPT" or "Tie",
        "reason": "상세 이유"
    }}
    """
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        response_format={ "type": "json_object" }
    )
    return json.loads(response.choices[0].message.content)

In [ ]:
# 6. 실행 및 결과 리포트
system_instruction = "당신은 반려동물 관리 및 건강 상담을 제공하는 '다정하고 전문적인 펫 상담사'입니다. 사용자의 질문에 대해 공감하며 전문적인 조언을 제공하세요."

# 평가용 데이터셋 로드 (학습 데이터 중 일부 샘플링)
dataset_path = "/workspace/train_dataset.jsonl"
results = []

with open(dataset_path, 'r', encoding='utf-8') as f:
    all_data = [json.loads(line) for line in f]
    eval_samples = random.sample(all_data, 5)

print("Comparing and Evaluating models...")
for i, data in enumerate(eval_samples):
    q = data['input']
    gt = data['output']
    
    print(f"[{i+1}/5] Processing: {q[:20]}...")
    
    gemma_ans = generate_gemma_response(system_instruction, q)
    gpt_ans = get_gpt_mini_response(system_instruction, q)
    judgment = evaluate_with_gpt4o(q, gt, gemma_ans, gpt_ans)
    
    results.append({
        "질문": q,
        "Gemma 2": gemma_ans,
        "GPT-4o-mini": gpt_ans,
        "Gemma 점수": judgment['gemma_score'],
        "GPT 점수": judgment['gpt_score'],
        "승자": judgment['winner'],
        "이유": judgment['reason']
    })

df = pd.DataFrame(results)
print("\n--- 🏆 Evaluation Result ---")
print(f"Gemma 2 Avg Score: {df['Gemma 점수'].mean():.2f}")
print(f"GPT-4o-mini Avg Score: {df['GPT 점수'].mean():.2f}")
display(df)